# Staged history matching with pestpp-ies, through the API

This notebook does something the `pestpp-ies` executable cannot: it runs a few iterations
against one set of observations, **changes which observation weights**, and
carries on with the same ensemble for more iterations.

That is a normal workflow in groundwater modeling that is more bespoke in practice. 
You rarely want every observation fighting for influence from
iteration one - heads and fluxes are on different scales, they constrain different things, and
throwing them all in at once tends to let the loudest group dominate. And we guess at which 
observations should get which weights before adjusting parameters and learning more about which 
obs can and cannot be fit...

The reason this needs the API is not that pest++ cannot do it - it is that the intervention
happens between iterations, and the built-in loop has no gap to put this action in and, more importantly
this type of intervention is very problem specific - it is impossible to design a generic scheme
to cover all the possible usecases (well, not impossible, but it would be very ugly!). 

**What this demonstrates**

1. Run iterations with only the head observations weighted.
2. Switch the streamflow (`gage`) observations **on**, through an explicit call.
3. Optionally **reinflate** the parameter ensemble at the same moment, so it has the spread to
   respond to the new data.
4. Carry on iterating, and watch phi decompose by group.

The model is the MODFLOW 6 Freyberg synthetic - small, fast, and the standard teaching case.

## Setup

Two knobs at the top. `REINFLATE` is the interesting one and is discussed properly further
down - leave it `True` the first time, then re-run the notebook with it `False` and compare.

In [1]:
import os
import shutil
import sys

import numpy as np
import pandas as pd
import pyemu

sys.path.insert(0, os.path.join("..", "python"))
from pestpp import Ies

N_REALS   = 10        # small so the notebook runs in well under a minute

BENCH = os.path.join("..", "benchmarks")
# the model binary has to be findable by the forward run
os.environ["PATH"] += os.pathsep + os.path.abspath(
    os.path.join(BENCH, "test_bin",
                 "win" if os.name == "nt" else
                 ("mac" if sys.platform == "darwin" else "linux")))

workdir = "staged_master"
if os.path.exists(workdir):
    shutil.rmtree(workdir)
shutil.copytree(os.path.join(BENCH, "mf6_freyberg", "template"), workdir)
print("working directory:", workdir)

working directory: staged_master


## Split the observations into two stages

The Freyberg case has 36 weighted observations: 24 groundwater levels at two sites
(`trgw_*`) and 12 streamflow values at the gage. Stage one uses the heads; stage two adds the
gage.

Setting the gage weights to zero is what makes them *inactive* - and inactive means more than
"contributes nothing to phi". An observation with zero weight is left out of the active set
entirely: it gets no column in the weights ensemble, and - the part that is easy to forget -
**no noise realizations are drawn for it**, because there was nothing to draw them from.
Switching it back on later has to put all of that back, which is why it needs a real call
rather than just an assignment.

In [2]:
pst_file = "freyberg6_run_ies.pst"
pst = pyemu.Pst(os.path.join(workdir, pst_file))

groups = pst.observation_data.loc[pst.nnz_obs_names, "obgnme"]
head_obs = [n for n in pst.nnz_obs_names if groups[n].startswith("trgw")]
flux_obs = [n for n in pst.nnz_obs_names if groups[n] == "gage"]
print("stage 1 - heads     :", len(head_obs))
print("stage 2 - streamflow:", len(flux_obs))

# remember what the flux weights were, so stage two can restore them rather than invent them
flux_weights = pst.observation_data.loc[flux_obs, "weight"].astype(float).to_dict()

pst.observation_data.loc[flux_obs, "weight"] = 0.0        # stage one: heads only
pst.control_data.noptmax = 1
pst.pestpp_options["ies_num_reals"] = N_REALS * 2
pst.pestpp_options["random_seed"] = 11
pst.pestpp_options["ies_no_noise"] = False

# This case ships with localization switched on, and localization is resolved against the
# ACTIVE observation set. A localizer built while the gage observations were switched off has
# no rows for them, so it has to be told to tolerate that - or, as here, left out of a demo
# that is about something else. See the closing notes.
pst.pestpp_options.pop("ies_localizer", None)
pst.pestpp_options.pop("ies_autoadaloc", None)

pst.write(os.path.join(workdir, pst_file), version=2)
print("wrote", pst_file)

stage 1 - heads     : 24
stage 2 - streamflow: 12
noptmax:1, npar_adj:8175, nnz_obs:24
wrote freyberg6_run_ies.pst


## Stage one: heads only

Nothing unusual here - open a session, initialize, iterate. The only thing worth noticing is
what `initialize()` decided the active set was.

In [3]:
history = []      # (stage, iteration, phi_mean) as we go

ies = Ies.from_pst(pst_file, workdir=workdir,ies_num_reals=50,ies_reinflate_num_reals=10)
ies.initialize()

#make these first few iters cheap
ies.set_option("ies_lambda_mults", "1.0")
ies.set_option("lambda_scale_fac", "1.0")

active = list(ies.weights_df(lower=True).columns)
print("realizations      :", ies.n_reals)
print("active observations:", len(active), "(all heads:",
      all(a.startswith("trgw") for a in active), ")")
print("initial phi        : {0:.4g}".format(ies.phi))
initial_phi = ies.phi
history.append(("stage 1", 0, ies.phi))

processing control file freyberg6_run_ies.pst
              starting serial run manager ...

realizations      : 10
active observations: 24 (all heads: True )
initial phi        : 653.8


In [4]:
for _ in range(10):
    step = ies.solve()
    history.append(("stage 1", step.iter, step.phi_mean))
    print("iteration {0}: phi_mean {1:.4g}".format(step.iter, step.phi_mean))
    if step.phi_mean < initial_phi / 10:
        print("mean phi < 1/10 * initial_phi, breaking") 
        break

iteration 1: phi_mean 106.2
iteration 2: phi_mean 71.41
iteration 3: phi_mean 61.69
mean phi < 1/10 * initial_phi, breaking


## The switch

This is the part the executable has no room for.

`set_obs_weights` here is doing more than assigning numbers. Because these observations were
at zero weight, they are being **activated**: they join the active set, gain a column in the
weights ensemble, and get noise realizations generated for them using the same draw the
initial ensemble used - same ordering, same grouping, same generator. Without that last part
they would be compared against nothing.

Note what is deliberately *not* coupled: changing the weight of an observation that was
already active does **not** touch its noise. Weights and noise are independent, and redrawing
noise on every weight change would make this iteration's phi incomparable with the last one's.
Only the off-to-on transition is structural, because only then is there no noise to preserve.

In [5]:
flux_weights

{'gage_1_20160131': 0.0043570137028,
 'gage_1_20160229': 0.0035933092582,
 'gage_1_20160331': 0.0034957090172,
 'gage_1_20160430': 0.0038145372013,
 'gage_1_20160531': 0.004575926053,
 'gage_1_20160630': 0.0065902201134,
 'gage_1_20160731': 0.0099806375631,
 'gage_1_20160831': 0.017123434276,
 'gage_1_20160930': 0.026124325666,
 'gage_1_20161031': 0.022379625589,
 'gage_1_20161130': 0.014472617807,
 'gage_1_20161231': 0.007512329611}

In [6]:
ies.noise_df()

obsnme,TRGW_2_2_9_20160131,TRGW_2_2_9_20160229,TRGW_2_2_9_20160331,TRGW_2_2_9_20160430,TRGW_2_2_9_20160531,TRGW_2_2_9_20160630,TRGW_2_2_9_20160731,TRGW_2_2_9_20160831,TRGW_2_2_9_20160930,TRGW_2_2_9_20161031,...,TRGW_2_33_7_20160331,TRGW_2_33_7_20160430,TRGW_2_33_7_20160531,TRGW_2_33_7_20160630,TRGW_2_33_7_20160731,TRGW_2_33_7_20160831,TRGW_2_33_7_20160930,TRGW_2_33_7_20161031,TRGW_2_33_7_20161130,TRGW_2_33_7_20161231
realization,,,,,,,,,,,,,,,,,,,,,
0,34.852353,34.768115,35.047661,34.915109,34.912882,34.813220,34.477658,34.415054,34.428712,34.376789,...,34.278896,34.277588,34.177246,34.030093,33.757430,33.686156,33.567497,33.627419,33.803410,34.034626
1,34.695712,35.051814,34.915800,34.946145,34.702267,34.756495,34.534835,34.499539,34.383452,34.429247,...,34.454780,34.360842,34.253761,34.129338,33.879662,33.689171,33.596788,33.556721,33.671900,33.935809
2,34.809618,34.864917,34.990545,34.843966,34.984314,34.721707,34.551673,34.479993,34.583900,34.354305,...,34.316330,34.402823,34.248696,34.056384,33.806995,33.676783,33.653145,33.610199,33.809439,33.886523
3,34.822131,34.985062,34.959187,34.989462,34.939237,34.831105,34.740957,34.454190,34.469782,34.429241,...,34.359785,34.370546,34.219943,34.013695,33.919021,33.660529,33.705104,33.601009,33.735698,33.890102
4,34.863879,34.997569,34.923141,34.870974,34.959827,34.619055,34.576950,34.413843,34.335948,34.525681,...,34.411263,34.324480,34.313576,34.011982,33.792594,33.704896,33.520333,33.576785,33.735499,33.854190
5,34.752445,34.952773,34.951031,34.872627,34.923920,34.712282,34.518079,34.413368,34.430533,34.387898,...,34.432237,34.435516,34.225221,34.143920,33.779580,33.667089,33.536541,33.625997,33.672864,33.993748
6,34.852947,34.929082,35.047798,35.070233,34.795735,34.750813,34.601505,34.499526,34.456463,34.237757,...,34.300349,34.359679,34.224595,34.047588,33.866072,33.675301,33.748501,33.603255,33.683130,33.857666
7,34.734617,34.954867,34.901477,34.783678,34.806274,34.741003,34.655375,34.507797,34.440219,34.323829,...,34.460494,34.307273,34.253711,33.997024,33.750201,33.671533,33.516489,33.646611,33.635385,34.030813
8,34.772613,35.022505,35.018154,34.961064,34.952272,34.733001,34.421995,34.478549,34.298114,34.459920,...,34.423936,34.390895,34.229458,33.960545,33.736382,33.704532,33.607186,33.555402,33.760889,33.952797


In [7]:
ies.set_obs_weights(flux_weights)          # <-- the whole switch

...initializing observation noise covariance matrix
...obscov loaded  from observation weights
...drawing observation noise realizations for activated observations:  12
...re-initializing localizer for the newly activated observations
...number of non-zero weighted observations increased from 24 to 36


Now check that the noise ensemble has been expanded:

In [8]:
ies.noise_df()

obsnme,GAGE_1_20160131,GAGE_1_20160229,GAGE_1_20160331,GAGE_1_20160430,GAGE_1_20160531,GAGE_1_20160630,GAGE_1_20160731,GAGE_1_20160831,GAGE_1_20160930,GAGE_1_20161031,...,TRGW_2_33_7_20160331,TRGW_2_33_7_20160430,TRGW_2_33_7_20160531,TRGW_2_33_7_20160630,TRGW_2_33_7_20160731,TRGW_2_33_7_20160831,TRGW_2_33_7_20160930,TRGW_2_33_7_20161031,TRGW_2_33_7_20161130,TRGW_2_33_7_20161231
realization,,,,,,,,,,,,,,,,,,,,,
0,1549.749708,1704.213397,1763.624235,1738.901098,1260.362270,952.721935,763.418587,356.847670,210.735267,275.680528,...,34.278896,34.277588,34.177246,34.030093,33.757430,33.686156,33.567497,33.627419,33.803410,34.034626
1,1784.565021,1172.460972,1580.275942,1873.430152,1740.244751,1194.545557,599.274614,367.252858,234.334534,274.553701,...,34.454780,34.360842,34.253761,34.129338,33.879662,33.689171,33.596788,33.556721,33.671900,33.935809
2,1780.638981,1971.744050,2032.559254,2076.660934,1631.751244,1018.619704,651.924649,489.178130,244.206917,264.615888,...,34.316330,34.402823,34.248696,34.056384,33.806995,33.676783,33.653145,33.610199,33.809439,33.886523
3,1012.515654,1645.829433,1623.631963,1765.986883,1428.767104,975.448528,599.698188,461.438375,292.909435,274.459336,...,34.359785,34.370546,34.219943,34.013695,33.919021,33.660529,33.705104,33.601009,33.735698,33.890102
4,1698.141108,2122.114197,2101.053165,1551.176137,1393.737358,860.343963,725.519070,315.755121,300.825665,274.242922,...,34.411263,34.324480,34.313576,34.011982,33.792594,33.704896,33.520333,33.576785,33.735499,33.854190
5,1329.286738,1853.494040,1907.144036,1861.041423,1394.928401,1067.139324,585.861494,331.114006,272.387918,274.467233,...,34.432237,34.435516,34.225221,34.143920,33.779580,33.667089,33.536541,33.625997,33.672864,33.993748
6,1659.375563,2168.158257,1761.039541,1724.026317,1406.641598,811.055207,640.254943,477.059028,261.279242,305.733669,...,34.300349,34.359679,34.224595,34.047588,33.866072,33.675301,33.748501,33.603255,33.683130,33.857666
7,1255.408667,1223.315568,1287.625994,1461.736058,1550.010417,1284.973596,769.541893,418.604226,285.025793,383.155536,...,34.460494,34.307273,34.253711,33.997024,33.750201,33.671533,33.516489,33.646611,33.635385,34.030813
8,1631.994031,1529.361445,2265.402024,1502.047244,1227.220878,1011.724634,784.658086,341.199464,253.490828,374.395779,...,34.423936,34.390895,34.229458,33.960545,33.736382,33.704532,33.607186,33.555402,33.760889,33.952797


In [9]:
ies.weights_df()

obsnme,GAGE_1_20160131,GAGE_1_20160229,GAGE_1_20160331,GAGE_1_20160430,GAGE_1_20160531,GAGE_1_20160630,GAGE_1_20160731,GAGE_1_20160831,GAGE_1_20160930,GAGE_1_20161031,...,TRGW_2_33_7_20160331,TRGW_2_33_7_20160430,TRGW_2_33_7_20160531,TRGW_2_33_7_20160630,TRGW_2_33_7_20160731,TRGW_2_33_7_20160831,TRGW_2_33_7_20160930,TRGW_2_33_7_20161031,TRGW_2_33_7_20161130,TRGW_2_33_7_20161231
realization,,,,,,,,,,,,,,,,,,,,,
0,0.004357,0.003593,0.003496,0.003815,0.004576,0.00659,0.009981,0.017123,0.026124,0.02238,...,15.0,15.0,15.0,15.0,15.0,15.0,15.0,15.0,15.0,15.0
1,0.004357,0.003593,0.003496,0.003815,0.004576,0.00659,0.009981,0.017123,0.026124,0.02238,...,15.0,15.0,15.0,15.0,15.0,15.0,15.0,15.0,15.0,15.0
2,0.004357,0.003593,0.003496,0.003815,0.004576,0.00659,0.009981,0.017123,0.026124,0.02238,...,15.0,15.0,15.0,15.0,15.0,15.0,15.0,15.0,15.0,15.0
3,0.004357,0.003593,0.003496,0.003815,0.004576,0.00659,0.009981,0.017123,0.026124,0.02238,...,15.0,15.0,15.0,15.0,15.0,15.0,15.0,15.0,15.0,15.0
4,0.004357,0.003593,0.003496,0.003815,0.004576,0.00659,0.009981,0.017123,0.026124,0.02238,...,15.0,15.0,15.0,15.0,15.0,15.0,15.0,15.0,15.0,15.0
5,0.004357,0.003593,0.003496,0.003815,0.004576,0.00659,0.009981,0.017123,0.026124,0.02238,...,15.0,15.0,15.0,15.0,15.0,15.0,15.0,15.0,15.0,15.0
6,0.004357,0.003593,0.003496,0.003815,0.004576,0.00659,0.009981,0.017123,0.026124,0.02238,...,15.0,15.0,15.0,15.0,15.0,15.0,15.0,15.0,15.0,15.0
7,0.004357,0.003593,0.003496,0.003815,0.004576,0.00659,0.009981,0.017123,0.026124,0.02238,...,15.0,15.0,15.0,15.0,15.0,15.0,15.0,15.0,15.0,15.0
8,0.004357,0.003593,0.003496,0.003815,0.004576,0.00659,0.009981,0.017123,0.026124,0.02238,...,15.0,15.0,15.0,15.0,15.0,15.0,15.0,15.0,15.0,15.0


## Reinflation

In [10]:

spread_before = ies.par_df(lower=True).std().mean()
ies.reinflate(factor=1.0,num_reals=20)
spread_after = ies.par_df(lower=True).std().mean()
print("mean parameter std: {0:.4g} -> {1:.4g}  ({2:.1f}x)".format(
    spread_before, spread_after, spread_after / spread_before))


mean parameter std: 2.846 -> 7.222  (2.5x)


In [11]:
ies.obs_df()

obsnme,GAGE_1_20151231,GAGE_1_20160131,GAGE_1_20160229,GAGE_1_20160331,GAGE_1_20160430,GAGE_1_20160531,GAGE_1_20160630,GAGE_1_20160731,GAGE_1_20160831,GAGE_1_20160930,...,TRGW_2_9_1_20170331,TRGW_2_9_1_20170430,TRGW_2_9_1_20170531,TRGW_2_9_1_20170630,TRGW_2_9_1_20170731,TRGW_2_9_1_20170831,TRGW_2_9_1_20170930,TRGW_2_9_1_20171031,TRGW_2_9_1_20171130,TRGW_2_9_1_20171231
realization,,,,,,,,,,,,,,,,,,,,,
0,1422.894032,1760.052667,1923.864979,1877.056310,1712.832684,1588.688292,1326.461825,1104.849487,901.430195,891.482730,...,34.839129,34.826821,34.807266,34.760457,34.697938,34.643855,34.615685,34.605009,34.641738,34.682823
1,2004.837772,2196.058726,2362.680534,2436.132872,2306.113835,2082.849560,1810.714080,1534.894501,1335.117773,1189.383108,...,36.264722,36.230054,36.172901,36.111508,36.051176,35.976451,35.944707,35.983377,36.033633,36.134839
2,2266.627168,2426.567255,2527.315811,2482.468257,2394.711515,2069.567792,1723.922166,1397.921199,1231.679225,1127.497377,...,35.464154,35.388042,35.292048,35.183810,35.075739,34.953421,34.906386,34.954571,35.074852,35.328529
3,2220.309250,2505.807601,2561.592118,2577.278133,2314.544614,2017.993724,1636.818967,1336.773920,1112.249566,954.494664,...,34.886687,34.860095,34.805152,34.746604,34.647443,34.567887,34.516175,34.565568,34.719616,34.765559
4,463.243382,740.901089,934.070762,1072.071566,1129.354528,979.497050,825.332658,645.330389,507.495287,444.039179,...,34.703013,34.697465,34.652705,34.549878,34.440822,34.346144,34.293189,34.339381,34.493477,34.711128
5,1160.898380,1358.291042,1466.511948,1507.776340,1378.868542,1215.325693,978.423274,790.299083,593.183438,499.863216,...,34.454242,34.449845,34.428700,34.393487,34.336631,34.282450,34.243846,34.243316,34.334393,34.403330
6,1393.515895,1631.403026,1929.985890,2105.062846,2008.245411,1861.283653,1555.031273,1284.724097,993.806518,856.152966,...,34.616654,34.646791,34.541196,34.416013,34.293948,34.201612,34.167636,34.253472,34.507802,34.734620
7,1720.759329,1907.811745,1899.813812,1830.170394,1638.476992,1440.247310,1210.131490,1030.815568,875.996078,848.041844,...,34.835871,34.806653,34.774212,34.696639,34.604540,34.528396,34.507209,34.605469,34.748344,34.903137
8,1512.447516,1827.834373,1950.577447,2059.325180,1986.866387,1818.015394,1600.171958,1372.480563,1146.088957,1061.594943,...,35.683830,35.594805,35.479462,35.273467,35.062438,34.917280,34.826576,34.803505,34.822680,34.999438


In [12]:
ies.par_df()

parnme,NPF_K33_0_000_000,NPF_K33_0_000_001,NPF_K33_0_000_002,NPF_K33_0_000_003,NPF_K33_0_000_004,NPF_K33_0_000_005,NPF_K33_0_000_006,NPF_K33_0_000_007,NPF_K33_0_000_008,NPF_K33_0_000_009,...,WELFLX_2_9_16_22,WELFLX_2_9_16_23,WELFLX_2_9_16_24,WELFLX_2_9_16_3,WELFLX_2_9_16_4,WELFLX_2_9_16_5,WELFLX_2_9_16_6,WELFLX_2_9_16_7,WELFLX_2_9_16_8,WELFLX_2_9_16_9
realization,,,,,,,,,,,,,,,,,,,,,
0,0.030000,0.161183,0.402985,0.252950,0.177340,0.167067,0.164073,0.283674,0.253623,0.205943,...,266.745182,244.190240,154.950101,153.478341,222.392200,273.978694,473.755362,443.710184,479.832588,322.233762
1,0.396273,0.110864,0.122949,1.100367,1.156028,1.421786,0.214353,0.660984,0.275633,0.459673,...,165.234246,112.555264,124.025591,138.223754,287.817810,381.879451,449.342238,510.305600,505.253105,473.566350
2,0.273811,0.318080,0.705176,0.657107,0.541909,2.594884,0.462505,0.340385,0.227635,0.184265,...,351.499767,244.190240,154.950101,133.295710,208.308478,267.252261,369.373219,510.305600,406.580393,385.837476
3,0.239440,0.219247,0.119043,0.303755,0.038164,0.127042,0.143084,0.333262,0.144995,0.779852,...,351.499767,238.431229,142.293834,187.515842,229.067944,354.957220,473.755362,226.224964,245.582888,288.727886
4,0.401215,0.106017,0.102010,0.243829,0.226479,0.194843,0.096899,0.168572,0.171067,0.114299,...,351.499767,161.103232,103.853739,130.557587,151.506092,142.953168,167.617068,186.038560,202.125020,237.555840
5,0.310490,0.313876,0.190212,0.947661,1.430577,1.821218,0.217178,0.352141,0.360380,0.242848,...,334.439668,197.471037,154.950101,167.888856,165.538980,179.420140,473.755362,462.105953,505.253105,478.808873
6,0.266131,0.198941,0.075953,0.067040,0.093585,0.100119,0.064540,0.030000,0.036305,0.113562,...,234.650993,171.841859,101.049081,187.515842,221.573745,293.786438,375.414450,459.298446,396.957614,465.803901
7,0.074900,0.053367,0.072499,0.208640,0.072950,0.196400,0.090908,0.149245,0.037503,0.055814,...,351.499767,240.634191,154.950101,187.515842,287.817810,343.750663,473.755362,422.771287,505.253105,410.481564
8,0.253469,0.858442,0.270524,0.373564,0.543438,0.944151,0.619852,1.261987,1.994276,1.647424,...,351.499767,105.382067,92.028920,36.741735,59.575781,69.027710,121.887709,193.235813,369.671665,217.258143


## Stage two

Recompute phi first, so the jump caused by the new observations is visible on its own rather
than mixed in with the effect of an iteration.

In [13]:
ies.update_phi()
print("phi with the new observations included: {0:.4g}".format(ies.phi))
history.append(("switch", history[-1][1], ies.phi))
#increase the solver power here too
ies.set_option("ies_lambda_mults",(0.1,1,10))
ies.set_option("ies_bad_phi_sigma",1.4)

for _ in range(5):
    step = ies.solve()
    history.append(("stage 2", step.iter, step.phi_mean))
    print("iteration {0}: phi_mean {1:.4g}".format(step.iter, step.phi_mean))

phi with the new observations included: 1290
iteration 4: phi_mean 120.4
iteration 5: phi_mean 55.37
iteration 6: phi_mean 44.49
iteration 7: phi_mean 39.19
iteration 8: phi_mean 36.29


## What happened, by group

`phi_group_df` decomposes phi the way the tools' own "observation group phi summary" does, but
as a frame. This is the payoff: the gage contribution appears from nothing at the switch, and
both groups are then fitted together.

In [14]:
grp = ies.phi_group_df(lower=True)
summary = grp.mean().sort_values(ascending=False)
print("mean phi contribution by group, final ensemble:")
print(summary.to_string())

print()
print("phi history:")
print(pd.DataFrame(history, columns=["stage", "iteration", "phi"]).to_string(index=False))

mean phi contribution by group, final ensemble:
trgw_2_33_7    14.046188
trgw_2_2_9     11.374267
gage           10.865981

phi history:
  stage  iteration         phi
stage 1          0  653.836352
stage 1          1  106.176880
stage 1          2   71.406268
stage 1          3   61.685420
 switch          3 1290.111332
stage 2          4  120.379454
stage 2          5   55.373505
stage 2          6   44.491955
stage 2          7   39.188574
stage 2          8   36.286436


In [15]:
ies.finalize()
ies.close()
print("done - outputs are in", workdir)

done - outputs are in staged_master


## Notes and caveats

**Localization is switched off in this notebook, on purpose.** A localizer is resolved against
the active observation set when it is built, so one built while the gage observations were
inactive has no rows for them. The library re-reads the localizer when observations are
activated, which handles the file-based case; `ies_autoadaloc` rebuilds its own matrix during
the solve and does not yet cope with the active set growing underneath it. If you need
localization *and* staging today, use a file-based localizer with
`ies_localizer_forgive_missing` set, and check the `.rec` file.

**Phi is not comparable across the switch.** Stage one's phi is measured over 24 observations
and stage two's over 36, so the jump at the switch is arithmetic, not a failure. Compare
within a stage, or compare per-group contributions.

**Reinflation discards information.** It deliberately puts back variance the assimilation had
removed, which is the point - but if the ensemble had genuinely converged on the right answer,
you are throwing some of that away. It is a tool for when you believe the narrowing was
premature, which is exactly what bringing in unseen data implies.

**On this case, over two iterations, it is close to a wash** - and that is worth saying rather
than letting the notebook imply otherwise. Running it both ways gives roughly:

| | at the switch | iteration 3 | iteration 4 |
|---|---|---|---|
| `REINFLATE = True` | 1234 | 210 | 165 |
| `REINFLATE = False` | 238 | 214 | 166 |

The reinflated ensemble starts much worse - it has just been given back its prior spread, so
of course it fits less well - and then improves faster, ending in the same place. Two
iterations is not enough to separate them. What the reinflated run *has* bought is spread: it
reaches iteration 4 with an ensemble that can still move, where the un-reinflated one is
running out of room. Whether that pays for itself depends on how many iterations follow and
how much the new data really disagrees with the old, which is a judgement about your problem
rather than something a default can make for you.